In [15]:
import os
import cv2
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# Poore workspace mein automatic search jo actual image folders pakde
data_dir = None
for root, dirs, files in os.walk("."):
    if len(dirs) > 5 and not ".ipynb_checkpoints" in root:
        # Check karo kya iske andar subfolders mein .jpg images hain
        sample_sub = os.path.join(root, dirs[0])
        if os.path.isdir(sample_sub):
            try:
                sub_files = os.listdir(sample_sub)
                if any(f.lower().endswith(('.jpg', '.jpeg', '.png')) for f in sub_files):
                    data_dir = root
                    break
            except Exception:
                pass

print(f"Detected Dataset Directory: {data_dir}")

X = []
y = []

if data_dir:
    classes = sorted([d for d in os.listdir(data_dir) if os.path.isdir(os.path.join(data_dir, d))])[:4]
    print(f"Selected Food Categories: {classes}")

    for category in classes:
        cat_path = os.path.join(data_dir, category)
        class_idx = classes.index(category)
        count = 0
        for img_name in os.listdir(cat_path):
            if count >= 80:  # Fast local run limit
                break
            if img_name.lower().endswith(('.jpg', '.jpeg', '.png')):
                try:
                    img_path = os.path.join(cat_path, img_name)
                    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
                    if img is not None:
                        img_resized = cv2.resize(img, (64, 64))
                        X.append(img_resized.flatten())
                        y.append(class_idx)
                        count += 1
                except Exception:
                    pass

X = np.array(X) / 255.0
y = np.array(y)

print(f"Total processed samples: {len(X)}")

if len(X) > 0:
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    print("Model training shuru ho rahi hai...")
    model = RandomForestClassifier(n_estimators=50, random_state=42)
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    print(f"\nModel Accuracy: {accuracy_score(y_test, y_pred) * 100:.2f}%")
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred, target_names=classes[:len(set(y))]))
else:
    print("Error: Images load nahi ho paayin.")

Detected Dataset Directory: .\leapGestRecog\00
Selected Food Categories: ['01_palm', '02_l', '03_fist', '04_fist_moved']
Total processed samples: 320
Model training shuru ho rahi hai...

Model Accuracy: 100.00%

Classification Report:
               precision    recall  f1-score   support

      01_palm       1.00      1.00      1.00        20
         02_l       1.00      1.00      1.00        12
      03_fist       1.00      1.00      1.00        20
04_fist_moved       1.00      1.00      1.00        12

     accuracy                           1.00        64
    macro avg       1.00      1.00      1.00        64
 weighted avg       1.00      1.00      1.00        64

